In [ ]:
## Plotting cv of cluster sizes. Smaller number mean that it better reflects the number of ancestral populations.

import altair as alt
import polars as pl

from itertools import product
import textwrap

# Variables to configure
path = "/master/abagwell/variant-analysis/results/rhesus_old/admixture/ADMIXTURE/unsupervised"
dataset = "U42_WES"
subset = "founders2"
# CV errors found in the .log files from ADMIXTURE runs
Ks = [2, 3, 4, 5, 6, 7]
seeds = [899, 900, 901, 902, 903, 904, 905, 906, 907, 908]

# Find CV error of each pair of K and seed
pairs = []
for K, seed in product(Ks, seeds):
    with open(f"{path}/{dataset}.{subset}.SNP.autosomal.seed{seed}.{K}.log") as f:
        for line in f:
            if line.startswith("CV error (K={}):".format(K)):
                pairs.append([K, line.strip().split()[-1]])

# Create dataframe
df = pl.DataFrame(pairs, schema=["K", "CV Error"], orient="row", schema_overrides={"K": pl.Int8, "CV Error": pl.Float64})

min = df.select("CV Error").min().item()
max = df.select("CV Error").max().item()

lineplot = alt.Chart(df).mark_line(color="orange").encode(
    alt.X('K:O'),
    alt.Y('CV Error:Q', aggregate='mean').scale(domain=[min, max]),
)
dotplot = alt.Chart(df).mark_circle(color="black").encode(
    alt.X('K:O'),
    alt.Y('CV Error:Q', title="CV Error").scale(domain=[min, max]),
    tooltip=[
        alt.Tooltip('CV Error')
    ]
)

description = """
Unsupervised admixture using ADMIXTURE v1.3.0 across K-clusters with K from 2 to 7.
Ten predetermined seeds were run for each K-cluster, represented by the black dots.
The orange line represents the mean CV error of the seeds at each K-cluster.
"""

(lineplot + dotplot).properties(
    height=120,
    width=80,
    title=["Cross-validation", "Error Across K-Clusters"],
).properties(
    title=alt.TitleParams(
    textwrap.wrap(description, width=30),
    orient='bottom',
    anchor='start',
    fontSize=10,
    fontWeight='normal'
    )
)#.save(f"{path}/{dataset}.{subset}.cv_error.unsupervised.v2.html")

In [ ]:
df.group_by("K").agg(pl.col("CV Error").mean())